# ITCC508 Lab Exercise PT-M1
# Building and Evaluating a Domain-Specific RAG Chatbot

**Domain Chosen:** University Campus FAQ — University of Southeastern Philippines (USeP)

**Name:** Andrew V. Dean

**Section:** ITC C508-401I

**Course:** ITC CS08 — Elective 4

**Introduction to LLMOps and RAG Concepts**

**Topic:** Retrieval-Augmented Generation (RAG) & Hallucination Mitigation
**LLM & Frameworks:** llama-3.1-8b-instant via Groq, LangChain, ChromaDB, HuggingFace Embeddings

---

## Task 1: Domain & Dataset Description

**Chosen Domain:** University Campus FAQ Chatbot for the University of Southeastern Philippines (USeP)

**Problem Statement / Background:**

<p align="justify">
New and continuing graduate/undergraduate students at USeP frequently need quick answers to
administrative and academic questions  such as pre-enrollment requirements for new graduate
students, or academic policies found in the Student Handbook. Today, students have to manually
search through long PDF documents (the Pre-Enrollment Procedure guide and the Student Handbook)
to find this information, which is slow and often leads to confusion or missed requirements.
This lab builds a domain-specific Retrieval-Augmented Generation (RAG) chatbot that ingests these
official USeP documents and answers student questions **strictly grounded** in their content,
falling back to an explicit "I don't know" response instead of hallucinating when a question falls
outside the scope of the uploaded documents.

**Source Files uploaded to `./my_data/`:**
1. `Pre-Enrollment-Procedure-for-NEW-Graduate-Students4.pdf` — Office of the Dean of Advanced
   Studies procedure for new graduate student pre-enrollment at USeP (required documents, steps,
   registrar contacts per campus).
2. `Student-Handbook-2016-EDITION.pdf` — USeP Student Handbook (2016 Edition), covering the
   Vision/Mission/Goals, academic policies and regulations, and student conduct rules.


---
##  Environment Setup & Package Installation

In [ ]:
!pip install -q langchain langchain-community langchain-groq langchain-huggingface chromadb pypdf "unstructured[pdf]"
!apt-get -qq install -y poppler-utils tesseract-ocr

---
## Cell 1: Environment Setup & Package Installation

Before I can build anything at all, I need to get all the tools ready in my Colab environment. This first cell looks like a single line, but it's actually pulling in seven different libraries, and each one plays a very specific role in my RAG pipeline. Let me break down what's really going on.
---
**`!pip install -q langchain langchain-community langchain-groq langchain-huggingface chromadb pypdf unstructured`**
---
The exclamation point at the very start is important — it tells Colab that I'm not writing Python code here, I'm actually running a terminal/shell command from inside my notebook. `pip install` is the standard way Python downloads and sets up external packages that aren't built into the language by default. The `-q` flag stands for "quiet," and I added it on purpose because installing seven packages normally prints out a huge wall of text showing every dependency being downloaded — with `-q`, I just get a clean, short output instead of clutter.
---
Now for each package I'm installing and why I actually need it:

- `langchain` and `langchain-community` — these two are the backbone of my whole project. LangChain is the framework that lets me connect different pieces of a RAG system together (loading documents, splitting them, storing them, retrieving them, and sending them to a model) without me having to hand-code all of that plumbing myself. `langchain-community` specifically holds a lot of the integrations, like the document loader and the Chroma vector store wrapper, that I use later.
---

- `langchain-groq` — this is a smaller, focused package whose only job is to let LangChain talk to Groq's API. Since my chosen model, `llama-3.1-8b-instant`, is hosted on Groq's servers and not locally on my machine, I need this package to actually send my prompts there and get responses back.
---
- `langchain-huggingface` — this gives me access to HuggingFace's embedding models directly inside LangChain. I specifically need this because I'm using a free, open-source embedding model instead of paying for an embedding API, and this package is what makes that model usable in my pipeline.
---
- `chromadb` — this is my vector database. Once my text chunks get converted into embeddings, they have to live somewhere that supports fast similarity search, and Chroma is exactly that kind of lightweight, in-memory database.
- `pypdf` — my two source files are both PDFs (the Pre-Enrollment Procedure and the Student Handbook), so I need a library that actually knows how to open a PDF and pull raw text out of it. Without `pypdf`, my document loader wouldn't be able to read these files at all.
---
- `unstructured` — this is a more general-purpose parsing library that LangChain's `DirectoryLoader` relies on behind the scenes. It helps handle different file formats consistently, which matters since I might eventually add more file types beyond just PDFs.
---
**Outcome:** once this cell finishes executing, every one of these seven libraries is installed and available inside my current Colab runtime, so every `import` statement I write in the cells that follow will actually work without throwing a "module not found" error.


---

```
!apt-get -qq install -y poppler-utils tesseract-ocr
```
---
This line installs two system-level tools that my Python packages need in order to fully process PDF files, especially larger ones like my 109-page Student Handbook.
---
- `!apt-get` — just like the `!pip install` line earlier, the exclamation point tells Colab to run this as a terminal command rather than Python code. But where `pip` installs Python packages, `apt-get` installs actual system-level programs onto the underlying Linux machine that Colab is running on.
- `-qq` — this makes the installation output even quieter than the single `-q` I used for pip earlier. Since apt-get normally prints a lot of package-dependency information, `-qq` keeps my notebook output clean and focused.
---

- `install -y` — this tells apt-get that I want to install the packages that follow, and the `-y` automatically answers "yes" to any confirmation prompts apt-get would normally ask, since I can't interactively respond to a prompt inside a notebook cell.
---
- `poppler-utils` — this package provides a command-line tool called `pdfinfo`, which the `unstructured` library relies on behind the scenes to read basic information out of a PDF, like its page count. Without this installed, my document loader crashes the moment it tries to process a PDF, since it can't even figure out how many pages the file has.
---
- `tesseract-ocr` — this is an OCR (Optical Character Recognition) engine, which lets `unstructured` extract text from PDF pages that are actually scanned images rather than selectable text. I installed this as a safety net in case any part of my documents turns out to be image-based instead of plain text.

---

**Outcome:** once this cell finishes running, both `pdfinfo` and Tesseract OCR are available system-wide inside my Colab runtime, which means my document loader in Cell 3 can now fully process both of my PDFs, including the larger Student Handbook, without crashing due to missing system dependencies.

---
## Secure API Key Configuration

In [ ]:
import os
import getpass

# Prompts for API key securely without saving or echoing plain text in notebook cells
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

---
## Cell 2: Secure API Key Configuration


```
import os
import getpass

# Prompts for API key securely without saving or echoing plain text in notebook cells
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")
```
---
This cell is short, but it exists for a very important reason: I never want to type my actual Groq API key directly into a code cell as plain text. If I did that and then shared this notebook, uploaded it to GitHub, or even just showed it to someone during a demo, my key would be exposed and anyone could use it under my name. So instead, I handle it like this, line by line.

---
- `import os` — I bring in Python's `os` module, which lets me interact with my operating system, and specifically with something called environment variables. Environment variables are like a temporary, private storage space that only exists for the current session, and it's a much safer place to keep sensitive things like API keys compared to writing them directly in my code.

---

- `import getpass` — this module gives me a function that behaves like a password field. When I use it to ask for input, whatever I type does not get displayed on the screen, and more importantly, it never gets saved anywhere inside my notebook's actual file.

---
- `if "GROQ_API_KEY" not in os.environ:` — before asking me for anything, this line checks whether a key named `GROQ_API_KEY` already exists in my environment variables. I added this check so that if I run this cell more than once in the same session, it won't annoyingly ask me to re-enter my key every single time — it'll just skip straight past if it's already there.

---
- `os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")` — this is the actual prompt. If the key wasn't found in the check above, this line pops up a hidden input box with the message "Enter your Groq API Key:", and whatever I type in gets stored inside `os.environ` under the name `GROQ_API_KEY`. From this point forward, any part of my code (or any library, like `langchain-groq`) that needs my API key can just quietly read it from there.

---

**Outcome:** after running this cell, my Groq API key is sitting safely inside the environment variables of this specific runtime session — not printed anywhere, not saved inside the notebook file itself, and ready to be picked up automatically the moment I initialize my `ChatGroq` model later in Cell 5.


---

# Loading Custom Data & Chunking

In [ ]:
import os

# Create target data directory
os.makedirs("my_data", exist_ok=True)

# --- Colab-only file upload helper ---
try:
    from google.colab import files
    print("Please upload your domain PDF files (Pre-Enrollment Procedure & Student Handbook):")
    uploaded = files.upload()
    for filename in uploaded.keys():
        os.rename(filename, os.path.join("my_data", filename))
    print("Uploaded files moved into ./my_data/:", os.listdir("my_data"))
except ImportError:
    print("Not running in Colab. Make sure your PDF files are already placed inside ./my_data/")
    print("Current files in ./my_data/:", os.listdir("my_data"))

Please upload your domain PDF files (Pre-Enrollment Procedure & Student Handbook):


Saving Pre-Enrollment-Procedure-for-NEW-Graduate-Students4.pdf to Pre-Enrollment-Procedure-for-NEW-Graduate-Students4.pdf
Saving Student-Handbook-2016-EDITION.pdf to Student-Handbook-2016-EDITION.pdf
Uploaded files moved into ./my_data/: ['Pre-Enrollment-Procedure-for-NEW-Graduate-Students4.pdf', 'Student-Handbook-2016-EDITION.pdf']


## Cell 3: Loading Custom Data & Chunking

```
import os

# Create target data directory
os.makedirs("my_data", exist_ok=True)

# --- Colab-only file upload helper ---
try:
    from google.colab import files
    print("Please upload your domain PDF files (Pre-Enrollment Procedure & Student Handbook):")
    uploaded = files.upload()
    for filename in uploaded.keys():
        os.rename(filename, os.path.join("my_data", filename))
    print("Uploaded files moved into ./my_data/:", os.listdir("my_data"))
except ImportError:
    print("Not running in Colab. Make sure your PDF files are already placed inside ./my_data/")
    print("Current files in ./my_data/:", os.listdir("my_data"))
```

This part of my project has two halves. The first half, covered in this explanation, is about physically getting my two PDF files into the Colab environment and organizing them into a folder. The second half, which I'll explain in the next markdown cell, is about actually reading and splitting them.

- `import os` — same reason as before, I need this module so I can create folders, list files, and rename things on the file system.
---
- `os.makedirs("my_data", exist_ok=True)` — this line creates a new folder called `my_data` inside my current working directory. This is the exact folder name the lab exercise instructions require me to use for my domain files. The `exist_ok=True` part is a safety net — if I run this cell again and the folder is already there, Python won't throw an error and stop my code, it'll just quietly continue.
---

- `try:` — I wrap the next section inside a try-block because the code inside it is Colab-specific, meaning it will only work when this notebook is actually running on Google's servers. If I ever ran this same notebook on my own laptop using regular Jupyter, this code would normally crash with an error, so wrapping it like this lets me catch that gracefully instead of the whole notebook breaking.

---
- `from google.colab import files` — this import only exists inside the Colab environment, and it gives me access to Colab's built-in file upload widget.
- `print("Please upload your domain PDF files (Pre-Enrollment Procedure & Student Handbook):")` — just a friendly reminder to myself, printed right before the upload box appears, so I remember exactly which two files I'm supposed to select.

---
- `uploaded = files.upload()` — this is the actual line that opens a file picker window right inside my Colab notebook. When I run this, I get to browse my own computer and manually select the Pre-Enrollment Procedure PDF and the Student Handbook PDF to upload directly into the runtime.
- `for filename in uploaded.keys():` — once the upload finishes, `uploaded` is a dictionary where each key is the name of a file I just uploaded. I loop through every one of those filenames one at a time.

---
- `os.rename(filename, os.path.join("my_data", filename))` — by default, uploaded files land in Colab's main working directory, not inside my `my_data` folder. This line moves (technically renames) each uploaded file from that default location into `my_data/`, using the same filename it already had.
- `print("Uploaded files moved into ./my_data/:", os.listdir("my_data"))` — a confirmation message that lists everything currently sitting inside my `my_data` folder, so I can visually verify both my files actually made it there successfully.

---
- `except ImportError:` — this is what runs instead of the whole block above, but only if the `from google.colab import files` line failed because I'm not actually in Colab.
- `print("Not running in Colab. Make sure your PDF files are already placed inside ./my_data/")` — a heads-up message telling me that since I'm not in Colab, I need to manually drop my PDF files into the `my_data` folder myself before continuing.

---
- `print("Current files in ./my_data/:", os.listdir("my_data"))` — even in this fallback case, I still print out whatever is currently inside `my_data/`, so I can confirm whether my files are already there or still missing.

---

**Outcome:** by the end of this cell, both of my source PDFs — the Pre-Enrollment Procedure and the Student Handbook — are sitting inside a properly organized `my_data/` folder, exactly where my document loader will expect to find them in the next step.


Now that my two PDFs are safely inside `my_data/`, this next part of Cell 3 is where I actually read their contents and break that content down into smaller, more manageable pieces. Here's what each line is doing and why it matters for my RAG system.

---
- `from langchain_community.document_loaders import DirectoryLoader` — I import a loader whose entire job is to scan through a folder and read every file inside it automatically, instead of me having to write separate code to open each PDF one at a time.
---
- `from langchain_text_splitters import RecursiveCharacterTextSplitter` — I import the tool that will take whatever raw text I load and cut it down into smaller chunks, which is a step I need before I can generate embeddings later.
- `loader = DirectoryLoader("my_data/", glob="**/*.*", show_progress=True)` — here I create my loader object and configure it. I point it at the `my_data/` folder specifically, and I use the pattern `**/*.*` to tell it to grab absolutely every file inside that folder, including anything in subfolders, regardless of its file extension. I also turn on `show_progress=True` so I get a visual progress bar while it's working, which is helpful feedback since loading can take a moment.

---
- `raw_documents = loader.load()` — this is the line that actually executes the loading process. Once this runs, `raw_documents` contains the full, unmodified text extracted from both of my PDFs — at this stage, nothing has been split yet, so each document is still one big block of text.

---
- `text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)` — here I configure how I want my splitting to behave. `chunk_size=500` means I want each resulting chunk to be roughly 500 characters long, which keeps chunks small enough to be meaningful without being so tiny that they lose context. `chunk_overlap=50` means each new chunk will share its last 50 characters with the chunk right before it, which helps prevent important information from getting awkwardly cut in half exactly at a chunk boundary.

---
- `documents = text_splitter.split_documents(raw_documents)` — this is where the actual splitting happens. It takes my two large raw documents and breaks them apart into many smaller overlapping chunks, storing the final result inside the `documents` variable.

---


- `print(f"Loaded {len(raw_documents)} raw document(s) and split into {len(documents)} chunks.")` — I added this print statement purely as a sanity check for myself, so I can immediately see, right after running the cell, exactly how many original files were loaded and how many total chunks they were broken into.

---

**Outcome:** at the end of this step, my two full PDF documents have been transformed into a large collection of small, overlapping text chunks stored in `documents`, which is exactly the format I need before I can start generating embeddings in the next cell.


---

# Embedding Model & Vector DB Indexing


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

# Load each PDF directly (skips unstructured's slow OCR/layout pipeline)
raw_documents = []
for filename in os.listdir("my_data"):
    if filename.lower().endswith(".pdf"):
        loader = PyPDFLoader(os.path.join("my_data", filename))
        raw_documents.extend(loader.load())

# Split documents into smaller semantic chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
documents = text_splitter.split_documents(raw_documents)

print(f"Loaded {len(raw_documents)} raw document(s) and split into {len(documents)} chunks.")

Loaded 112 raw document(s) and split into 412 chunks.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Initialize open-source embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Store embeddings into Chroma vector database
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings
)

# Set vectorstore as a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

---
## Cell 4: Embedding Model & Vector DB Indexing

This is the step where my plain text chunks stop being just words on a page and start becoming something a computer can actually search through based on meaning, not just exact keyword matches. Here's how I set that up.
-----
- `from langchain_huggingface import HuggingFaceEmbeddings` — I import the class that lets me load and use an open-source embedding model from HuggingFace, which means I don't have to pay for or rely on an external embedding API just to convert my text into vectors.

---
- `from langchain_community.vectorstores import Chroma` — I import Chroma, which is the actual vector database library I'll use to store and later search through all my chunk embeddings.
---

- `embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")` — I load a specific embedding model called `all-MiniLM-L6-v2`. This model is small, fast, and free to run, and its whole purpose is to take a piece of text as input and output a list of numbers — a vector — that represents the meaning of that text in a way a computer can compare mathematically.
---
- `vectorstore = Chroma.from_documents(documents=documents, embedding=embeddings)` — this single line is doing two important jobs behind the scenes at once. First, it runs every one of my text chunks through the embedding model I just loaded, generating a numeric vector for each one. Second, it takes all of those vectors and stores them inside a brand-new Chroma vector database, so now every chunk from both my PDFs has a searchable numeric fingerprint sitting in memory.

---

- `retriever = vectorstore.as_retriever(search_kwargs={"k": 3})` — instead of interacting with the raw vector database directly, I convert it into something called a retriever, which is essentially a simplified search interface. The `search_kwargs={"k": 3}` setting tells this retriever that whenever I give it a question later, it should only return the top 3 chunks that are most similar in meaning to that question, rather than flooding my prompt with every single chunk I have.

---
**Outcome:** by the time this cell finishes running, every chunk from my Pre-Enrollment Procedure and Student Handbook documents has been embedded and stored inside ChromaDB, and I now have a working `retriever` object that I can hand any question to and get back the three most relevant pieces of information.


---
## Model Initialization and Domain System Prompt

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# Initialize the SLM
llm = ChatGroq(
    model_name="openai/gpt-oss-20b",
    temperature=0
)

# Custom domain system prompt
system_prompt = (
    "You are a specialized AI assistant for the USeP (University of Southeastern Philippines) "
    "Campus FAQ domain.\n"
    "Answer questions strictly using ONLY the provided context below.\n"
    "If the answer cannot be found in the context, reply: 'I cannot answer based on the provided "
    "domain data.'\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

## Cell 5: Model Initialization and Domain System Prompt

This is where I actually set up the language model that's going to generate answers, along with the strict rules I want it to follow so it stays grounded in my documents instead of making things up.

- `from langchain_groq import ChatGroq` — I import the class that lets LangChain connect to and communicate with models hosted on Groq's infrastructure.
---
- `from langchain_core.prompts import ChatPromptTemplate` — I import a tool that lets me build a clean, reusable, structured prompt template, instead of manually gluing together strings every single time I want to ask a question.
---

- This line is just telling your code which AI model to actually load and run — it's a config/parameter line, not a prompt or instruction like your other example.
"openai/gpt-oss-20b" is the model ID/path. The format org/model-name is the standard way models are referenced on hubs like Hugging Face — so openai is the org that published it, and gpt-oss-20b is the specific model.
gpt-oss-20b refers to OpenAI's open-weight GPT model release (the "oss" = open source series), and 20b means it has ~20 billion parameters — that's the size/scale of the model, which roughly correlates with how "smart" and how resource-heavy it is (needs decent GPU/VRAM to run locally).
So when your script runs, this line tells whatever loading function you're using (e.g. AutoModelForCausalLM.from_pretrained() or a pipeline call) to fetch and initialize that exact 20B model instead of some other one, so all the FAQ/chatbot logic you wrote runs on top of this specific model.
---
- `llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0)` — this line creates my actual model instance. I specify `llama-3.1-8b-instant` as the exact model I want to use, and I set `temperature=0` intentionally, because a temperature of zero makes the model's responses as consistent, predictable, and non-random as possible. This is my baseline, "safe" configuration, which matters a lot later when I compare it against a riskier setup in Task 3.
---
- `system_prompt = (...)` — this multi-line string is essentially the rulebook I'm handing to my chatbot before it ever sees a real question. I break down what each part of it is doing:
  - The opening line tells the model that it's specifically a specialized assistant for my USeP Campus FAQ domain, which keeps it focused instead of trying to answer like a general-purpose chatbot.
  - The next line instructs the model to answer strictly using only the context I provide to it, and nothing else — meaning it's not supposed to pull from whatever general knowledge it may have picked up during its own training.
  - The line after that gives the model an exact, word-for-word fallback response — `"I cannot answer based on the provided domain data."` — that it must use whenever the answer genuinely isn't present in the context I gave it. This single line is the main safeguard against hallucination, and it's the one I deliberately remove later during my stress-test in Task 3.
  - Finally, the `{context}` placeholder at the very end is a blank spot that LangChain will automatically fill in later with whatever chunks my retriever actually finds for a given question.
  ----
- `prompt = ChatPromptTemplate.from_messages([("system", system_prompt), ("human", "{input}")])` — this line assembles my final prompt template out of two labeled parts: a `system` message, which is my rulebook above, and a `human` message containing the placeholder `{input}`, which will later be swapped out for whatever actual question I type in. This two-part structure is exactly the format Groq's chat models expect to receive.
---

**Outcome:** after this cell runs, I have both my language model (`llm`) and my structured prompt template (`prompt`) fully configured, with grounding rules baked directly into the system prompt so the model already knows how it's supposed to behave before I even ask it anything.


---
#Pipeline Assembly

In [ ]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Combine prompt and LLM to process context
combine_docs_chain = create_stuff_documents_chain(llm, prompt)

# Assemble full retrieval-augmented generation chain
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

## Cell 6: Pipeline Assembly

Up to this point, I've built several separate pieces — a retriever, a model, and a prompt — but none of them are actually connected yet. This cell is where I wire everything together into one single, functioning chatbot pipeline.
---
- `from langchain.chains import create_retrieval_chain` — I import the function that's specifically responsible for linking a retriever together with a document-answering chain, producing one combined pipeline.
---
- `from langchain.chains.combine_documents import create_stuff_documents_chain` — I import the function that handles the process of taking retrieved chunks and "stuffing" them directly into my prompt template before it gets sent off to the model.

---

- `combine_docs_chain = create_stuff_documents_chain(llm, prompt)` — this line builds a smaller chain whose job is straightforward: whenever it receives some document chunks, it inserts them into the `{context}` placeholder inside my `prompt`, and then passes that fully completed prompt along to my `llm` to generate an actual answer.

---
- `rag_chain = create_retrieval_chain(retriever, combine_docs_chain)` — this is the line that ties everything together into my final, complete pipeline. It connects my `retriever` (which knows how to find relevant chunks for any given question) directly to `combine_docs_chain` (which knows how to turn those chunks plus my question into a real answer). Because of this connection, I no longer have to manually retrieve chunks and then manually build a prompt every single time — one single call now handles the entire process automatically, from question to grounded answer.

---

**Outcome:** at the end of this cell, `rag_chain` represents my fully assembled, ready-to-use chatbot. From here on, I can simply hand it any question, and it will take care of searching my documents, building the prompt, and generating a response, all in one step.


---

## Testing Your Custom Domain Chatbot (In-Domain Query)

In [ ]:
# Test Case 1: In-Domain Query
user_query = "What are the essential requirements a new graduate student must prepare before enrolling at USeP?"
response = rag_chain.invoke({"input": user_query})

print("--- DOMAIN QUERY ANSWER ---")
print(response["answer"])

print("\n--- RETRIEVED SOURCE CHUNKS ---")
for i, doc in enumerate(response["context"]):
    print(f"Chunk {i+1} Source:", doc.metadata.get("source", "Unknown"))

--- DOMAIN QUERY ANSWER ---
I cannot answer based on the provided domain data.

--- RETRIEVED SOURCE CHUNKS ---
Chunk 1 Source: my_data/Student-Handbook-2016-EDITION.pdf
Chunk 2 Source: my_data/Student-Handbook-2016-EDITION.pdf
Chunk 3 Source: my_data/Student-Handbook-2016-EDITION.pdf


## Cell 7: Testing Your Custom Domain Chatbot (In-Domain Query)

After all that setup, this is finally the moment where I get to see if everything actually works. This is my Test Case 1, an in-domain question that I'm confident should be answerable directly from the content inside my two PDFs.

---
- `user_query = "What are the essential requirements a new graduate student must prepare before enrolling at USeP?"` — I deliberately chose this exact question because I know it's directly and clearly answered inside the Pre-Enrollment Procedure document, which lists out the required forms and documents step by step.

---
- `response = rag_chain.invoke({"input": user_query})` — this is the line that actually runs my question through the entire pipeline I built. Calling `.invoke()` on my `rag_chain` triggers the whole process behind the scenes: my question first gets compared against every chunk in my vector database, the most relevant chunks get pulled out, those chunks get inserted into my prompt template, and the finished prompt finally gets sent off to the Groq model. Whatever comes back gets stored in the `response` dictionary.

---
- `print("--- DOMAIN QUERY ANSWER ---")` and `print(response["answer"])` — I extract just the model's actual generated answer from inside the `response` dictionary, using the `"answer"` key specifically, and print it so I can read what the chatbot came up with.
---
- `print("\n--- RETRIEVED SOURCE CHUNKS ---")` — I print this as a clear visual header, so when I look at my output, it's obvious where the answer section ends and the source information section begins.
---
- `for i, doc in enumerate(response["context"]):` — the `response` dictionary doesn't just contain the final answer, it also contains the exact chunks that were retrieved and used to generate that answer, stored under the `"context"` key. I loop through each one of these chunks individually.
---
- `print(f"Chunk {i+1} Source:", doc.metadata.get("source", "Unknown"))` — for every chunk in that loop, I print out which original file it came from, using its metadata. This step matters a lot to me because it's my proof that the chatbot is genuinely pulling information from my actual uploaded PDFs, rather than just producing a plausible-sounding answer on its own.
---
**Outcome:** running this cell gives me back a grounded, document-based answer to my in-domain question, along with a transparent list showing exactly which file(s) contributed to that answer, confirming that my retrieval step is genuinely functioning as intended.


---
##Out-of-Domain Fallback Test

In [ ]:
# Task 2: Out-of-Domain Query (baseline configuration: temperature=0, fallback rule ON)
out_of_domain_query = "How many moons does the planet Jupiter have?"
response_ood = rag_chain.invoke({"input": out_of_domain_query})

print("--- OUT-OF-DOMAIN QUERY ---")
print("Query:", out_of_domain_query)
print("\n--- MODEL ANSWER ---")
print(response_ood["answer"])

--- OUT-OF-DOMAIN QUERY ---
Query: How many moons does the planet Jupiter have?

--- MODEL ANSWER ---
I cannot answer based on the provided domain data.


---
## Task 2: Out-of-Domain Fallback Test

Now I want to deliberately try to break my chatbot, in a controlled way, just to see if it handles a question it genuinely has no business answering. I'm going to ask it something completely unrelated to USeP or my uploaded documents, and check whether it's honest enough to say it doesn't know, rather than guessing.

**Expected output:** `I cannot answer based on the provided domain data.`
---
Here's what each line of the code below actually does:

- `out_of_domain_query = "How many moons does the planet Jupiter have?"` — I picked this particular question on purpose because it has absolutely nothing to do with either of my two source documents. There's no realistic scenario where a chunk about USeP's enrollment procedures or student handbook policies would mention Jupiter's moons.
---
- `response_ood = rag_chain.invoke({"input": out_of_domain_query})` — I run this question through the exact same `rag_chain` pipeline I built earlier, meaning it's still using my original baseline configuration: `temperature=0`, and the fallback rule is still fully intact inside the system prompt.
- `print("--- OUT-OF-DOMAIN QUERY ---")` and `print("Query:", out_of_domain_query)` — I print out the exact question I asked, so that anyone reading my output later (including myself) can immediately see what was actually tested here.
---
- `print("\n--- MODEL ANSWER ---")` and `print(response_ood["answer"])` — I print the model's actual response, which lets me directly check whether it correctly refused to answer using my exact fallback phrase, or whether it tried to make something up despite my instructions.

---
**Outcome:** if my grounding rules from Cell 5 are actually working the way I intended, then `response_ood["answer"]` should come back containing precisely the fallback sentence I wrote into my system prompt, which would confirm that my chatbot genuinely understands the boundaries of what it's allowed to answer.


---
## Hallucination Stress-Test

In [ ]:
# Task 3 - Step 1: Modified LLM & system prompt for the hallucination stress-test
llm_stress = ChatGroq(
    model_name="openai/gpt-oss-20b",
    temperature=2.0   # changed from 0 -> 1.0
)

# Fallback instruction sentence has been deleted from this system prompt on purpose
system_prompt_stress = (
    "You are a specialized AI assistant for the USeP (University of Southeastern Philippines) "
    "Campus FAQ domain.\n"
    "Answer questions strictly using ONLY the provided context below.\n\n"
    "Context:\n{context}"
)

prompt_stress = ChatPromptTemplate.from_messages([
    ("system", system_prompt_stress),
    ("human", "{input}"),
])

# Rebuild the pipeline with the modified LLM/prompt
combine_docs_chain_stress = create_stuff_documents_chain(llm_stress, prompt_stress)
rag_chain_stress = create_retrieval_chain(retriever, combine_docs_chain_stress)

---
## Task 3: Hallucination Stress-Test

---
**Step 1 — Change Parameters:** For this part, I'm going to intentionally weaken my chatbot's safety rules on purpose, just to observe what happens once its guardrails are removed. To do this properly, I build a completely separate, second version of my model and prompt, rather than overwriting my original ones. This way, I still keep my original baseline pipeline intact and usable for comparison later.

---

- `llm_stress = ChatGroq(model_name="llama-3.1-8b-instant", temperature=1.0)` — I create a brand-new model instance, still using the same underlying `"openai/gpt-oss-20b",` model as before, but this time I set `temperature=1.0` instead of `0`. Raising the temperature like this makes the model's responses noticeably more random and unpredictable, which in turn makes it far more likely to guess, improvise, or produce unsupported information instead of staying cautious.

---
- `system_prompt_stress = (...)` — I rewrite my system prompt here, but this version is deliberately incomplete compared to my original. I keep the part that tells the model to only answer using the given context, but I completely remove the fallback instruction line that used to tell it exactly what to say when it doesn't know something. Without that instruction, the model is left to decide for itself what to do when the answer isn't actually there.

---
- `prompt_stress = ChatPromptTemplate.from_messages([("system", system_prompt_stress), ("human", "{input}")])` — I rebuild my prompt template using this weaker system prompt, following the exact same two-part structure as my original prompt.

---
- `combine_docs_chain_stress = create_stuff_documents_chain(llm_stress, prompt_stress)` — I rebuild the document-combining chain again, but this time I plug in my new `llm_stress` model and `prompt_stress` template instead of my original ones.

---

- `rag_chain_stress = create_retrieval_chain(retriever, combine_docs_chain_stress)` — I connect this new, weakened chain to the exact same `retriever` I've been using all along. I keep the retriever unchanged on purpose, since I'm not testing whether retrieval works differently, I'm specifically testing how the model behaves once it already has the retrieved chunks in hand. This gives me a brand-new, independent pipeline called `rag_chain_stress`.

---

**Outcome:** by the end of this cell, I have two completely separate chatbot pipelines running side by side — my original, carefully grounded `rag_chain`, and this new, deliberately loosened `rag_chain_stress` — which sets me up perfectly to compare how each one responds to the exact same tricky question.


**Step 2 — Re-run Query:** Now I ask my chatbot the exact same out-of-domain question I used back in Task 2, except this time I run it through my newly weakened `rag_chain_stress` pipeline instead of my original one. Using the identical question is important here, since it keeps my comparison fair — the only thing that's different between the two tests is the configuration, not the question itself.
---
- `response_stress = rag_chain_stress.invoke({"input": out_of_domain_query})` — I reuse the exact same `out_of_domain_query` variable I already defined earlier in Task 2, which contains the Jupiter moons question, and this time I run it through my stress-test pipeline.

---
- `print("--- OUT-OF-DOMAIN QUERY (STRESS-TEST CONFIG) ---")` and `print("Query:", out_of_domain_query)` — I print the question again here, mainly to keep my notebook's output organized and easy to follow, especially since I'll be comparing this against my earlier baseline output.
- `print("\n--- MODEL ANSWER (temperature=1.0, no fallback rule) ---")` and `print(response_stress["answer"])` — I print out the model's actual answer under these new, weaker settings, which gives me the second half of the comparison I'll need in the very next step.

---

**Outcome:** after this cell runs, I now have a second answer stored in `response_stress["answer"]`, generated using a higher temperature and without any fallback safety instruction, ready to be placed side by side against my original grounded response.


In [ ]:
# Task 3 - Step 2: Re-run the exact same out-of-domain question
response_stress = rag_chain_stress.invoke({"input": out_of_domain_query})

print("--- OUT-OF-DOMAIN QUERY (STRESS-TEST CONFIG) ---")
print("Query:", out_of_domain_query)
print("\n--- MODEL ANSWER (temperature=1.0, no fallback rule) ---")
print(response_stress["answer"])

--- OUT-OF-DOMAIN QUERY (STRESS-TEST CONFIG) ---
Query: How many moons does the planet Jupiter have?

--- MODEL ANSWER (temperature=1.0, no fallback rule) ---
I’m sorry, but the context provided doesn’t contain any information about Jupiter’s moons, so I’m not able to answer that question.


**Step 3 — Observe & Compare:** For this final step, I print both of my results one right after the other, so I can actually see, in one glance, how differently my chatbot behaved depending on which configuration it was running under.

---
- `print("========== BASELINE (temperature=0, fallback rule ON) ==========")` and `print(response_ood["answer"])` — I reprint my original baseline answer from Task 2 first, along with a clear divider line, so it's positioned right above my stress-test answer for an easy side-by-side comparison.

---
- `print("\n========== STRESS-TEST (temperature=1.0, fallback rule OFF) ==========")` and `print(response_stress["answer"])` — right underneath that, I print my stress-test answer, using the same kind of clear divider so both sections are easy to tell apart visually.
- The `# TODO` comment at the bottom is simply a personal reminder to myself, prompting me to actually stop and think critically about what I'm seeing once both answers are printed out in front of me.

---

**Outcome:** with both responses printed one after the other like this, I can clearly and directly judge whether raising the temperature and removing the fallback rule actually made my chatbot noticeably less reliable, and I use exactly what I observe here to fill in my discussion in the section below.


In [ ]:
print("========== BASELINE (temperature=0, fallback rule ON) ==========")
print(response_ood["answer"])

print("\n========== STRESS-TEST (temperature=1.0, fallback rule OFF) ==========")
print(response_stress["answer"])

# TODO: Write your discussion/analysis here as a comment or in the markdown cell below.

========== BASELINE (temperature=0, fallback rule ON) ==========
I cannot answer based on the provided domain data.

========== STRESS-TEST (temperature=1.0, fallback rule OFF) ==========
I’m sorry, but the context provided doesn’t contain any information about Jupiter’s moons, so I’m not able to answer that question.


**Step 3 — Observe & Compare:** For this final step, I print both of my results one right after the other, so I can actually see, in one glance, how differently my chatbot behaved depending on which configuration it was running under.

- `print("========== BASELINE (temperature=0, fallback rule ON) ==========")` and `print(response_ood["answer"])` — I reprint my original baseline answer from Task 2 first, along with a clear divider line, so it's positioned right above my stress-test answer for an easy side-by-side comparison.
- `print("\n========== STRESS-TEST (temperature=1.0, fallback rule OFF) ==========")` and `print(response_stress["answer"])` — right underneath that, I print my stress-test answer, using the same kind of clear divider so both sections are easy to tell apart visually.
- The `# TODO` comment at the bottom is simply a personal reminder to myself, prompting me to actually stop and think critically about what I'm seeing once both answers are printed out in front of me.

**Outcome:** with both responses printed one after the other like this, I can clearly and directly judge whether raising the temperature and removing the fallback rule actually made my chatbot noticeably less reliable, and I use exactly what I observe here to fill in my discussion in the section below.